# Multi-Period Planning

Up to T6 every model spanned a single planning window — a day, a week — and `Sizing` was the investment tool. Real planning problems span years: maybe demand grows, maybe fuel prices ramp, maybe carbon prices kick in. `periods=[...]` adds a horizon dimension and lets each period have its own dispatch.

You'll meet:

- `periods=[...]` — extra axis on every variable. Each period shares the timestep structure but has independent dispatch.
- `period_weights=[...]` — how much each period contributes to the objective. Default: inferred from period gaps (5-year gaps → weight 5 each). Override for NPV discounting.
- **`Sizing` shifts meaning** — in multi-period mode, the optimizer picks one size *per period independently*, with `effects_per_size` charged in each period.

That last point is the key change. In a single-period model, `Sizing` represented the investment decision. In multi-period, the same `Sizing` object is per-period — natural for things you re-decide each year:

- **Grid connection fee** — annual contracted capacity with a utility (kW × €/kW/year).
- **Short-term leases** — renting equipment by the period.
- **Annual tariff brackets** — capacity-based fees committed yearly.

For a *one-time built* asset (real CAPEX paid once, not an annualized rate), use `Investment` — T8 covers that explicitly.


In [ ]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from fluxopt import Carrier, Converter, Effect, Flow, Port, Sizing, optimize

## 1. Story: a bakery expanding production

A bakery runs a heat-driven workshop on natural gas. The plant operations team is locking in a 15-year planning view (2025, 2030, 2035): they are bringing on new ovens and a second shift, so workshop heat demand is set to grow ~50 % over the horizon.

Same gas → boiler → workshop heat system as T6, with three planning periods at five-year gaps. The 24-hour demand *shape* is the bakery's daily rhythm and stays fixed — its *amplitude* grows period-to-period as production ramps up.

The natural way to feed period-varying demand into `fixed_relative_profile` is a `pd.DataFrame` with timesteps as the index and periods as the columns:


In [ ]:
n = 24
timesteps = [datetime(2025, 1, 15) + timedelta(hours=h) for h in range(n)]
periods = [2025, 2030, 2035]

# Fixed daily shape — the bakery's rhythm doesn't change.
base_pattern = np.array(
    [10, 10, 8, 8, 8, 12, 25, 60, 70, 75, 75, 70, 65, 60, 55, 50, 45, 40, 30, 25, 20, 15, 12, 10],
    dtype=float,
)

# Production ramps: +20 % by 2030, +50 % by 2035.
growth = {2025: 1.0, 2030: 1.2, 2035: 1.5}

time_idx = pd.DatetimeIndex(timesteps, name='time')
demand = pd.DataFrame(
    {p: base_pattern * g for p, g in growth.items()},
    index=time_idx,
).rename_axis(columns='period')

peak = float(demand.values.max())
profile = demand / peak  # values in [0, 1]
demand.head()

In [ ]:
def build(**extra):
    return optimize(
        timesteps=timesteps,
        periods=periods,
        carriers=[Carrier(id='gas'), Carrier(id='heat')],
        effects=[Effect(id='cost', unit='EUR')],
        ports=[
            Port(
                id='gas_grid',
                imports=[Flow(carrier='gas', size=1000, effects_per_flow_hour={'cost': 0.10})],
            ),
            Port(
                id='workshop',
                exports=[Flow(carrier='heat', size=peak, fixed_relative_profile=profile)],
            ),
        ],
        converters=[
            Converter.boiler(
                'boiler',
                thermal_efficiency=0.9,
                fuel_flow=Flow(carrier='gas', size=300),
                thermal_flow=Flow(
                    carrier='heat', size=Sizing(size_min=20, size_max=200, effects_per_size={'cost': 1500})
                ),
            ),
        ],
        objective='cost',
        **extra,
    )


result = build()
result.effect_totals.to_dataframe(name='total').round(2)

`effect_totals` is now 2-D — `effect × period`. Per-period cost climbs as production grows: more heat, more gas, more annualized boiler capex.

The boiler's chosen size has a period axis — one size per period:


In [ ]:
result.solution['flow--size'].sel(flow='boiler(heat)').to_dataframe(name='size (kW)').round(1)

**This is the per-period nature of `Sizing` in action.** Each period chooses its own capacity to match its own peak demand: ~75 kW in 2025, ~90 kW in 2030, ~112.5 kW in 2035 — the same +20 %/+50 % growth as the demand. `effects_per_size=1500` is charged in *each* period, so the annualized boiler capex grows in step with capacity.

Contrast with T8 (Investment): one capacity decision spanning the horizon, sized to the *peak across all periods*, with one-time CAPEX paid in the chosen build period.


In [ ]:
# Default weights: inferred from period gaps. 5-year gaps → weight 5 per period.
pd.Series([5, 5, 5], index=pd.Index(periods, name='period'), name='weight (default)')

In [ ]:
print(f'Objective: {result.objective:.2f} EUR  (= sum of weight × per-period cost)')

## 2. NPV weights

Future euros are worth less today. Discount each period to present value with a 5 % rate over the five-year span. Override the global weights via `period_weights=`.


In [ ]:
r = 0.05
period_offsets = [0, 5, 10]  # year offsets of each period from today
npv_weights = [round(sum(1 / (1 + r) ** (y0 + y) for y in range(5)), 3) for y0 in period_offsets]

result_npv = build(period_weights=npv_weights)

pd.DataFrame(
    {
        'flat': [5, 5, 5],
        'NPV (r=5%)': npv_weights,
        'cost/period': result_npv.effect_totals.sel(effect='cost').values.round(2),
    },
    index=pd.Index(periods, name='period'),
)

In [ ]:
pd.Series(
    {
        'Flat (default)': round(result.objective, 2),
        'NPV (r=5%)': round(result_npv.objective, 2),
    },
    name='objective (EUR)',
)

NPV-weighted is lower because future periods are discounted. Per-period sizes don't change between flat and NPV runs — sizing is driven by demand, not by how much each period contributes to the objective. What changes is the *total* objective value.

Per-effect weights — say physical effects (CO₂, fuel kWh) keep flat weights while only `cost` is discounted — are configured on each `Effect` directly via `period_weights=`. See the API reference.

## Recap

`periods=[...]` adds a period axis to every variable. `period_weights=` (global on `optimize()`, or per-`Effect`) controls each period's contribution to the objective.

`Sizing` in multi-period decides capacity *per period independently*, charging `effects_per_size` each period — the right tool when each period has its own annualized cost (grid fees, leases, tariff brackets) and capacity can flex with demand. When you need a single capacity choice that spans the horizon, with one-time CAPEX paid in a chosen build period, that's `Investment` — next.


## 3. Periods are independent episodes

Some constraints link a timestep to the one before it: a storage level
(`E[t] = E[t-1] + charge - discharge`), a minimum-uptime window, a ramp limit.
Those links only make physical sense when `t-1` and `t` are actually adjacent
in real time. A maximal stretch of genuinely consecutive timesteps is called
an **episode**, and fluxopt never chains a constraint across an episode
boundary.

In a multi-period model the clock jumps between periods — the last hour of
2025 is *not* followed by the first hour of 2030 — so **each period is its own
episode**:

- storage recursion, uptime/downtime windows, startup detection and ramps all
  reset at each period start;
- `Storage.cyclic` closes the loop *within* each period (level at the period's
  last timestep equals its first);
- `Storage.prior_level` and `Flow.prior_rates` apply at *every* period start —
  each period sees the same pre-horizon history.

A **period**, by contrast, is a bookkeeping unit: it answers *which ledger a
timestep's costs land in* (per-period totals, `period_weights`, per-period
`Sizing`). Episode answers *did the clock jump*. Today the two coincide 1:1;
once representative-period aggregation lands, one period will contain several
episodes — the accounting stays per period while the coupling resets per
episode.

### Custom constraints on the flat time axis

All operational variables live on **one flat `time` dimension** spanning every
period, with a `time_period` coordinate mapping each timestep to its period.
Labels are real timestamps, so datetime features work directly —
`sel(time=slice('2030-06', '2030-08'))`, `groupby('time.dt.month')`. Two house
rules keep custom constraints correct:

1. **Weight by `dt`, always.** `groupby(...).sum()` counts timesteps, which
   silently diverges once resolutions differ per period. The dt-weighted form
   `(rate * dt).groupby(...).sum()` is exact on any grid mix.
2. **Labels are interval starts.** A selection finer than a period's
   resolution matches nothing — `time.dt.hour == 3` is vacuous on a 4-hourly
   grid — and a constraint built from it silently disappears. Select at or
   above the coarsest resolution your model uses.

Periods may also use entirely different grids (hourly 2030, 4-hourly 2050) by
passing `timesteps={2030: idx_a, 2050: idx_b}` — an advanced feature; see
`docs/design/time-index.md` for its semantics and caveats before reaching for
it.
